In [ ]:
import onep
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import scipy
from scipy import stats
import scipy.stats
% matplotlib inline

In [ ]:
#Multiple mice
import pickle

# Load the pickle file
with open('/Users/suthardr/Desktop/collection_cxta_fc.pkl', 'rb') as file:
    collection_cxta_fc = pickle.load(file)

with open('/Users/suthardr/Desktop/collection_cxtb_fc.pkl', 'rb') as file:
    collection_cxtb_fc = pickle.load(file)

In [ ]:
animal = 'astro3'

fc_traces = collection_cxta_fc.animals[animal].accepted_traces.T
fc_traces = fc_traces.to_numpy()
fc_tracesz = stats.zscore(fc_traces, axis=1)
fc_tracesz
# rejected_indices = collection_cxta_fc.animals[animal].rejected_inds

In [ ]:
#Grab the argmax
across_eta_, time = onep.eta_individual_cells(
    data=fc_traces,
    timestamps=collection_cxta_fc.animals[animal].Timestamps[:3603, ].to_numpy().squeeze(),
    events=[[120, 180, 240, 300], ],  #detected onsets
    window=28
)

In [ ]:
import numpy as np

# Initialize max_idxs based on the shape of across_eta_
max_idxs = np.zeros(across_eta_.shape[0], dtype=int)

# Iterate over each row of across_eta_ to find the maximum index
for i in range(across_eta_.shape[0]):
    idx = np.argmax(across_eta_[i])
    max_idxs[i] = idx

# Now, use max_idxs to index into the time array
resulting_times = time[max_idxs]
resulting_times

In [ ]:
#Grab the pred_mu
mu_fc = pd.read_csv("/Users/suthardr/Desktop/Rui_models/astrocyte_shot_Gaussian_noise3_origID.csv")

In [ ]:
animal_to_filter = [animal]

# Use .isin() to filter the DataFrame
mu_fc_animal = mu_fc[mu_fc['animal'].isin(animal_to_filter)]

mu_fc_animal

In [ ]:
plt.scatter(mu_fc_animal['pred_mu'],resulting_times, color='blue', alpha=0.7)
# Add a line of best fit (linear regression)
# First, fit a linear regression model
coefficients = np.polyfit(mu_fc_animal['pred_mu'], resulting_times, 1)
poly = np.poly1d(coefficients)

# Plot the line of best fit
plt.plot(mu_fc_animal['pred_mu'], poly(mu_fc_animal['pred_mu']), color='red')

# Add labels and title
plt.xlabel('pred_mu (mode)')
plt.ylabel('argmax (z-scored trace)')
plt.title('Astro3 All Cells: FC')

# Display the plot
plt.savefig("/Users/suthardr/Desktop/Rui_models/astro3_fc_pred_mu_vs_argmax_MODE_Zscore.svg")

In [ ]:
#Run spearman's correlation on x and y
rho, p = stats.spearmanr(mu_fc_animal['pred_mu'],resulting_times, axis=0)

#print Spearman rank correlation and p-value
print(rho)
print(p)